# Hypothesis: the initial projection preserves forest performance

We compare the same random forest trained on the original features `X` and on the initial projected representation `X₀`. The practical tolerance is five percentage points of accuracy loss. The test split is used only for the final report.

In [1]:
import sys
from pathlib import Path

notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / '_hypothesis_utils.py').exists():
    notebooks_dir = Path.cwd()
sys.path.insert(0, str(notebooks_dir))

import pandas as pd
from sklearn.metrics import accuracy_score
from _hypothesis_utils import (
    CLASSIFICATION_SEEDS, classification_data, forest_classifier,
    initial_projection, print_verdict,
)

In [2]:
rows = []
for seed in CLASSIFICATION_SEEDS[:2]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    direct = forest_classifier(seed).fit(X_train, y_train)
    direct_score = accuracy_score(y_test, direct.predict(X_test))
    for dimension in [16, 32, 64]:
        X_train_0, X_test_0, _ = initial_projection(
            X_train, X_test, dimension, seed
        )
        projected = forest_classifier(seed).fit(X_train_0, y_train)
        projected_score = accuracy_score(y_test, projected.predict(X_test_0))
        rows.append({
            'seed': seed, 'dimension': dimension,
            'accuracy_X': direct_score, 'accuracy_X0': projected_score,
            'loss': direct_score - projected_score,
        })
results = pd.DataFrame(rows)
display(results.round(3))
summary = results.groupby('dimension')['loss'].agg(['mean', 'max']).round(3)
display(summary)
tolerance = 0.05
supported = bool((summary['max'] <= tolerance).all())
print_verdict(
    'Initial projection preserves forest performance',
    supported,
    f'maximum observed accuracy loss is {summary["max"].max():.3f}; '
    f'predefined tolerance is {tolerance:.2f}',
)

,seed,dimension,accuracy_X,accuracy_X0,loss
0,0,16,0.761,0.711,0.050
1,0,32,0.761,0.750,0.011
2,0,64,0.761,0.782,-0.020
3,1,16,0.820,0.766,0.055
4,1,32,0.820,0.800,0.020
5,1,64,0.820,0.798,0.023


,mean,max
dimension,,
16,0.052,0.055
32,0.016,0.020
64,0.001,0.023


Initial projection preserves forest performance: NOT SUPPORTED — maximum observed accuracy loss is 0.055; predefined tolerance is 0.05


## Conclusion

**Hypothesis:** the initial projection preserves random-forest performance within a 0.05 absolute-accuracy tolerance.

**Experiment:** fit matched forests on the original and projected representations across two seeds and target dimensions 16, 32, and 64 using the difficult 120-feature classification task.

**Measure:** held-out test accuracy loss, defined as accuracy on `X` minus accuracy on `X₀`; the predefined tolerance was 0.05.

**Result:** not supported in this run: the maximum observed loss was 0.055, slightly above the tolerance.